# Phase 1 — Baseline Evaluation

Local holdout for **Baseline A** (Dragapult only) vs **Baseline B** (Dragapult + UCB1 search).

Opponent panel (research plan): Alakazam, Crustle, Spidops, Starmie.

Kaggle tasks (ladder submissions) are documented in `docs/phases/phase_01/online/KAGGLE_LOG.md`. Offline holdout: `docs/phases/phase_01/offline/HOLDOUT_LOG.md`.


In [3]:
!pip3 install -r ../requirements.txt

  Using cached kaggle_environments-1.32.2-py3-none-any.whl.metadata (828 bytes)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached gymnasium-1.2.0-py3-none-any.whl.metadata (9.9 kB)
  Using cached gymnax-0.0.8-py3-none-any.whl.metadata (19 kB)
  Using cached jax-0.10.2-py3-none-any.whl.metadata (13 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 3.4 MB/s  0:00:05 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 3.1 MB/s  0:00:00eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 3.6 MB/s  0:00:16m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 944.3/944.3 kB 3.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [4]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

NOTEBOOKS = Path.cwd() if (Path.cwd() / "env_paths.py").exists() else Path.cwd() / "notebooks"
if str(NOTEBOOKS) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS))

from env_paths import get_paths, setup_runtime

PATHS = get_paths()
setup_runtime(PATHS)

# Subprocess cells must use the project .venv (system Python lacks kaggle-environments).
VENV_PYTHON = PATHS.repo_root / ".venv" / "bin" / "python"
PYTHON = str(VENV_PYTHON if VENV_PYTHON.exists() else sys.executable)

print("repo_root:", PATHS.repo_root)
print("python:", PYTHON)
if VENV_PYTHON.exists() and Path(sys.executable).resolve() != VENV_PYTHON.resolve():
    print("Tip: select the project .venv kernel so notebook and subprocess use the same interpreter.")


repo_root: /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG
python: /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/.venv/bin/python


## 1. Build baseline variants


In [5]:
builder = NOTEBOOKS / "build_merged_agent.py"
subprocess.run([PYTHON, str(builder), "--variant", "all"], check=True, cwd=str(NOTEBOOKS))


Wrote /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/agents/main_baseline_a.py
Wrote /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/agents/main_baseline_b.py
Wrote /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/merged_agent_main.py
Synced /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/main.py


CompletedProcess(args=['/Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/.venv/bin/python', '/Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/build_merged_agent.py', '--variant', 'all'], returncode=0)

## 2. Refresh holdout opponent panel


In [6]:
extractor = NOTEBOOKS / "extract_holdout_panel.py"
subprocess.run([PYTHON, str(extractor)], check=True, cwd=str(NOTEBOOKS))


Wrote holdout panel under /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/holdout/panel


CompletedProcess(args=['/Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/.venv/bin/python', '/Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/extract_holdout_panel.py'], returncode=0)

## 3. Run holdout suite (40 games × 4 opponents × 2 baselines)


In [7]:
HOLDOUT_GAMES = 40
runner = NOTEBOOKS / "run_phase1_holdout.py"
subprocess.run(
    [PYTHON, str(runner), "--games", str(HOLDOUT_GAMES)],
    check=True,
    cwd=str(NOTEBOOKS),
)


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 41.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: OpenSpiel games skipped: 0.
Wrote results to /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/output/phase1
baseline_a   vs alakazam win_rate=0.200 (8/40)
baseline_a   vs crustle  win_rate=0.775 (31/40)
baseline_a   vs spidops  win_rate=0.850 (34/40)
baseline_a   vs starmie  win_rate=0.825 (33/40)
baseline_b   vs alakazam win_rate=0.150 (6/40)
baseline_b   vs crustle  win_rate=0.850 (34/40)
baseline_b   vs spidops  win_rate=0.750 (30/40)
baseline_b   vs starmie  win_rate=0.800 (32/40)


CompletedProcess(args=['/Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/.venv/bin/python', '/Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/run_phase1_holdout.py', '--games', '40'], returncode=0)

## 4. Summaries


In [8]:
import json

summary_path = NOTEBOOKS.parent / "docs/phases/phase_01/offline/results/phase1_holdout_summary_latest.json"
summaries = json.loads(summary_path.read_text(encoding="utf-8"))
df = pd.DataFrame(summaries)
display(df[["baseline", "opponent", "wins", "losses", "ties", "win_rate", "holdout_gate"]])


,baseline,opponent,wins,losses,ties,win_rate,holdout_gate
0,baseline_a,alakazam,8,32,0,0.200,holdout_fail
1,baseline_a,crustle,31,9,0,0.775,holdout_pass
2,baseline_a,spidops,34,6,0,0.850,holdout_pass
3,baseline_a,starmie,33,7,0,0.825,holdout_pass
4,baseline_b,alakazam,6,34,0,0.150,holdout_fail
5,baseline_b,crustle,34,6,0,0.850,holdout_pass
6,baseline_b,spidops,30,10,0,0.750,holdout_pass
7,baseline_b,starmie,32,8,0,0.800,holdout_pass


## 5. Kaggle (manual)

See `docs/phases/phase_01/online/KAGGLE_LOG.md` — submit Baseline A and Baseline B once each and record ladder ratings.
